# 3 - Closure and selection studies

Does `A = B*C/D` hold **bin by bin** in the fit observable? That is a much
stronger assumption than the integrated version a counting ABCD needs, and it is
what the shape fit rests on.

At the analysis selection it cannot be tested at all - region A holds 1-11
effective simulated events and most bins are empty. So the regions are loosened
**on purpose** until there is enough background to compare against. A loose point
is for *learning whether the factorisation holds*, not for setting limits, and
nothing here is an optimisation.

Command-line equivalent:

```bash
python sidm/scripts/abcd_plots.py closure --shapes <campaign>/shapes_iso_dphi.json
```

In [ ]:
import sys
from pathlib import Path

STUDY = Path.cwd()
REPO = STUDY.parents[2]
for p in (str(REPO), str(STUDY)):
    if p not in sys.path:
        sys.path.insert(0, p)

import datacard_tools as dt
import shape_tools as st
import plot_tools as pt
import numpy as np

CAMPAIGN = "sixd_inclusive_v1"
dt.use_campaign(CAMPAIGN)
PLOTS = STUDY / "plots" / CAMPAIGN / "closure"
PLANE = "iso_dphi"

## Read the files once

Slicing is cheap; the EOS read is not. Load the samples once and re-slice them
for every working point.

In [ ]:
import glob

cache = []
for f in sorted(glob.glob(dt.BKG_DIR + "/*.coffea")):
    cache.extend(dt.read_coffea(f).items())
print(f"loaded {len(cache)} background samples")

def grouped_for(selection, plane=PLANE):
    """Grouped background shapes for one Selection, from the cached samples."""
    per_sample = {}
    for sample, out in cache:
        per_channel = {}
        for ch in st.SIXD_HISTS:
            regions = st.region_shapes_6d(out, ch, plane, sample_name=sample,
                                          selection=selection)
            if regions:
                per_channel[ch] = regions
        if per_channel:
            per_sample[sample] = per_channel
    return st.group_shapes(per_sample)

## Working points

Loosening isolation alone achieves very little. The **displacement requirements**
are what cost the statistics, so turning them off is the move that reaches
hundreds of events.

In [ ]:
POINTS = {
    "nominal":       st.Selection("nominal"),
    "no disp":       st.Selection("no disp", apply_disp=False),
    "iso<0.5, no disp": st.Selection("iso<0.5, no disp", iso_cut=0.5, apply_disp=False),
    "iso<1.0, no disp": st.Selection("iso<1.0, no disp", iso_cut=1.0, apply_disp=False),
}

variants = {}
for label, sel in POINTS.items():
    variants[label] = grouped_for(sel)
    for ch in ("SR_2mu2e", "SR_4mu"):
        shape = st.total_background_shape(variants[label], ch, 0)
        tot, w2 = shape.sumw.sum(), shape.sumw2.sum()
        print(f"{label:18s} {ch:10s} A {tot:9.1f}  n_eff {tot**2/w2:6.1f}"
              f"  empty bins {int((shape.sumw <= 0).sum())}/{shape.nbins}")

## Per-bin closure

The ratio panel follows the data/MC convention: **points** carry the region-A
statistical error, the **band around unity** carries the prediction's. The
$\chi^2$ combines both, which is what a compatibility test needs.

In [ ]:
WP = "iso<0.5, no disp"
for ch in ("SR_2mu2e", "SR_4mu"):
    pt.closure_panel(variants[WP], ch, PLOTS, CAMPAIGN,
                     subtitle=POINTS[WP].describe(),
                     name=f"closure_{ch}")
    pt.shape_factorisation(variants[WP], ch, PLOTS, CAMPAIGN,
                           name=f"factorisation_{ch}")
print("done")

## Is a bias a constant factor?

If the ratio is flat in the observable it can be absorbed into a single transfer
factor; if it slopes, a constant correction would be wrong in the bins that
matter. The fit is done in log space, so the constant hypothesis is a horizontal
line and the intercept is the factor a correction would use.

In [ ]:
loose = {k: v for k, v in variants.items() if k != "nominal"}
for ch in ("SR_4mu", "SR_2mu2e"):
    rows = pt.closure_ratio_overlay(loose, ch, PLOTS, CAMPAIGN,
                                    name=f"closure_ratio_{ch}")
    print(f"-- {ch}")
    for r in rows:
        print(f"   {r['variant']:18s} factor {r['factor']:5.2f} +/- {r['factor_err']:4.2f}"
              f"   slope {r['slope']:+5.2f} +/- {r['slope_err']:4.2f}"
              f"   chi2 const {r['chi2_const']:5.1f}/{r['ndf_const']}")

## Does a cut distort the shape?

If it does not, the higher-statistics variant can measure a shape the tighter
one cannot measure for itself. Applying the displacement requirement to one
lepton jet at a time is the halfway house that keeps enough events to say
something.

In [ ]:
LEGS = {
    "neither":         st.Selection("neither", iso_cut=0.5, apply_disp=False),
    "leg 0 only":      st.Selection("leg0", iso_cut=0.5, disp_legs="leg0"),
    "leg 1 only":      st.Selection("leg1", iso_cut=0.5, disp_legs="leg1"),
    "both (analysis)": st.Selection("both", iso_cut=0.5, disp_legs="both"),
}
leg_variants = {k: grouped_for(v) for k, v in LEGS.items()}

for ch in ("SR_2mu2e", "SR_4mu"):
    stats = pt.compare_region_shapes(leg_variants, ch, 0, PLOTS, CAMPAIGN,
                                     reference="neither",
                                     name=f"shape_variants_{ch}")
    for label, (total, neff) in stats.items():
        print(f"{ch:10s} {label:18s} total {total:9.1f}  n_eff {neff:6.1f}")

Read `n_eff`, not the weighted yield: a variant can hold hundreds of events and
only ten effective ones when high-weight QCD dominates. "Consistent" at low
`n_eff` means *not excluded*, not *verified*.